<a href="https://colab.research.google.com/github/aj1365/CA-LMoETransUNet/blob/main/CA_LMoETransUNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout_prob=0.2):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout_prob)
        # Shortcut to match dimensions
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        shortcut = self.shortcut(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = self.bn2(self.conv2(x))
        x += shortcut
        return self.relu(x)


class MoEAttentionBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_dim, num_experts=4, dropout=0.4, class_2_focus=True):
        super(MoEAttentionBlock, self).__init__()
        self.num_heads = num_heads
        self.num_experts = num_experts
        self.scale = dim ** -0.5
        self.class_2_focus = class_2_focus
        self.qkv_proj = nn.ModuleList([nn.Linear(dim, dim * 3) for _ in range(num_experts)])
        self.output_proj = nn.ModuleList([nn.Linear(dim, dim) for _ in range(num_experts)])
        self.gate = nn.Linear(dim, num_experts)
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.ReLU(inplace=True),
            nn.Linear(mlp_dim, dim),
        )

    def forward(self, x, logits=None):
        B, L, D = x.shape  # B: Batch size, L: Sequence length, D: Embedding dimension
        x_norm = self.norm1(x)
        gate_values = self.gate(x_norm)

        if self.class_2_focus and logits is not None:
            class_2_logit = logits[:, 1]

            gate_values += 5.0 * class_2_logit.unsqueeze(-1)
        gate_probs = F.softmax(gate_values, dim=-1)  # Probabilities for each expert
        # Apply each expert's QKV projections and calculate attention
        expert_outputs = []
        for i in range(self.num_experts):
            qkv = self.qkv_proj[i](x_norm).chunk(3, dim=-1)  # Split into Q, K, V
            q, k, v = map(lambda t: t.contiguous().view(B, L, self.num_heads, -1).transpose(1, 2), qkv)
            # Compute scaled dot product attention for the i-th expert
            q = F.relu(q)
            k = F.relu(k)
            kv = torch.einsum('bnld,bnle->bnld', k, v)
            z = 1 / (torch.einsum('bnld,bnle->bnl', q, k) + 1e-6).unsqueeze(-1)
            attention = torch.einsum('bnld,bnle->bnle', q, kv) * z
            attention = attention.view(B * L, D)
            # Apply output projection for the i-th expert
            expert_outputs.append(self.dropout(self.output_proj[i](attention)))
        expert_outputs = [output.view(B, L, D) for output in expert_outputs]
        combined_output = sum(gate_probs[:, :, i:i+1] * expert_outputs[i] for i in range(self.num_experts))
        x = x + self.dropout(combined_output)
        x = x + self.dropout(self.mlp(self.norm2(x)))
        return x

class CloudWeighting(nn.Module):
    def __init__(self):
        super(CloudWeighting, self).__init__()
        self.sigmoid = nn.Sigmoid()
    def forward(self, features, cloud_prob):
        """
        Features are weighted by (1 - cloud_prob) to reduce cloud impact.
        Args:
            features (Tensor): Feature maps (N, C, H, W).
            cloud_prob (Tensor): Cloud probability maps (N, 1, H, W).
        """
        return features * (1 - self.sigmoid(cloud_prob))


class CALMoETransUNet(nn.Module):
    def __init__(self, in_channels=22, out_channels=2, embed_dim=128, num_heads=2, mlp_dim=256, transformer_depth=1, num_experts=4, class_2_focus=True):
        super(CALMoETransUNet, self).__init__()

        # Encoder (U-Net)
        self.enc1 = ResidualBlock(in_channels, 64, dropout_prob=0.2)
        self.enc2 = ResidualBlock(64, 128, dropout_prob=0.2)
        self.enc3 = ResidualBlock(128, 256, dropout_prob=0.2)
        self.enc4 = ResidualBlock(256, embed_dim, dropout_prob=0.2)
        self.flatten = nn.Flatten(2)
        self.transformer_blocks = nn.ModuleList([
            MoEAttentionBlock(embed_dim, num_heads, mlp_dim, num_experts, class_2_focus=class_2_focus) for _ in range(transformer_depth)
        ])
        # Decoder (U-Net)
        self.up4 = nn.ConvTranspose2d(embed_dim, 256, kernel_size=2, stride=2)
        self.dec4 = ResidualBlock(256, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ResidualBlock(128, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ResidualBlock(64, 64)
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)
        # Cloud Weighting
        self.cloud_weighting_pre = CloudWeighting()  # For pre-fire cloud maps
        self.cloud_weighting_post = CloudWeighting()  # For post-fire cloud maps
    def forward(self, x, logits=None):
        s1_data = x[:, :6, :, :]  # 6 channels (Sentinel-1)
        s2_pre = x[:, 6:14, :, :]  # 8 channels (pre-fire Sentinel-2)
        pre_cloud = x[:, 14:15, :, :]  # Pre-fire cloud map
        s2_post = x[:, 15:23, :, :]  # 8 channels (post-fire Sentinel-2)
        post_cloud = x[:, 23:24, :, :]  # Post-fire cloud map
        # Apply cloud weighting
        s2_pre_weighted = self.cloud_weighting_pre(s2_pre, pre_cloud)
        s2_post_weighted = self.cloud_weighting_post(s2_post, post_cloud)
        # Combine data for encoding
        s2_data = torch.cat([s2_pre_weighted, s2_post_weighted], dim=1)
        combined_data = torch.cat([s1_data, s2_data], dim=1)
        # Encoding Path
        enc1 = self.enc1(combined_data)
        enc2 = self.enc2(F.max_pool2d(enc1, 2))
        enc3 = self.enc3(F.max_pool2d(enc2, 2))
        enc4 = self.enc4(F.max_pool2d(enc3, 2))

        b, c, h, w = enc4.shape
        x = self.flatten(enc4).permute(0, 2, 1)  # (B, L, C)

        for transformer_block in self.transformer_blocks:
            x = transformer_block(x, logits=logits)

        x = x.permute(0, 2, 1).view(b, c, h, w)

        # Decoding Path
        dec4 = self.dec4(self.up4(x) + enc3)
        dec3 = self.dec3(self.up3(dec4) + enc2)
        dec2 = self.dec2(self.up2(dec3) + enc1)

        return self.final_conv(dec2)


in_channels = 22
out_channels = 2
embed_dim = 128
num_heads = 4
mlp_dim = 256
transformer_depth = 1  # Multiple Transformer blocks for efficiency
num_experts = 3  # Number of experts for MoE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CALMoETransUNet(in_channels, out_channels, embed_dim, num_heads, mlp_dim, transformer_depth, num_experts).to(device)

dummy_input = torch.randn(4, 24, 256, 256).to(device)

output = model(dummy_input)
output.shape


torch.Size([4, 2, 256, 256])